# 🏠 Predicting House Prices with Linear Regression

I wanted this notebook to read like a conversation with myself rather than a lab manual. The goal is simple: take a table of houses (area, location, rooms, age...) and see how well a plain old **linear regression** can guess what they sold for. Along the way I'll try to explain *why* I'm doing each step, not just *what* I'm typing.

**What's in here**

1. Load the data and poke around (EDA)
2. Decide which features I think matter, and say why
3. Deal with missing values and categorical columns
4. Look at correlations
5. Split into train / test (80/20)
6. Train a Linear Regression
7. Score it with MSE, RMSE and R²
8. Plot actual vs. predicted, and check the residuals
9. Read the coefficients like a human
10. Bonus: does Ridge or Lasso do any better?



## 0. Setting up

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, RidgeCV, LassoCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler



import kagglehub

# Download latest version
path = kagglehub.competition_download('house-prices-advanced-regression-techniques')

print("Path to competition files:", path)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

: 

## 1. Loading the data

The Kaggle file has 80 columns, which is a lot for a first linear model. I'm going to pick a small, readable set that maps onto the brief (area, location, rooms, age) plus a few extras that I strongly suspect matter (overall quality, garage, basement size). I'm renaming them to plain English so the rest of the notebook is easier to read.

| My name | Ames column | Meaning |
|---|---|---|
| `price` | `SalePrice` | what the house sold for (our target) |
| `area` | `GrLivArea` | above-ground living area, sq ft |
| `location` | `Neighborhood` | neighbourhood name |
| `rooms` | `TotRmsAbvGrd` | total rooms above ground |
| `age` | `YrSold - YearBuilt` | age of the house when sold |
| `quality` | `OverallQual` | overall material/finish rating, 1–10 |
| `garage` | `GarageCars` | garage capacity (cars) |
| `basement_area` | `TotalBsmtSF` | basement size, sq ft |
| `lot_frontage` | `LotFrontage` | street frontage, ft (this one has real missing values) |

In [ ]:
DATA_PATH = Path("train.csv")   # Kaggle file, OR house_prices_synthetic.csv renamed/pointed to here


def load_ames(path):
    raw = pd.read_csv(path)
    if "SalePrice" not in raw.columns:      # already in my simplified format (e.g. the synthetic CSV)
        return raw
    return pd.DataFrame({
        "price": raw["SalePrice"],
        "area": raw["GrLivArea"],
        "location": raw["Neighborhood"],
        "rooms": raw["TotRmsAbvGrd"],
        "age": raw["YrSold"] - raw["YearBuilt"],
        "quality": raw["OverallQual"],
        "garage": raw["GarageCars"],
        "basement_area": raw["TotalBsmtSF"],
        "lot_frontage": raw["LotFrontage"],
    })


def make_synthetic(n=1500, seed=RANDOM_STATE):
    """Stand-in data so the notebook runs without the Kaggle file."""
    rng = np.random.default_rng(seed)
    locations = {"Downtown": 90_000, "Riverside": 55_000, "Midtown": 30_000,
                 "Suburbs": 0, "Outskirts": -25_000}
    loc = rng.choice(list(locations), size=n, p=[0.15, 0.2, 0.25, 0.25, 0.15])
    area = rng.normal(1500, 450, n).clip(500, 4000)
    rooms = np.round(area / 250 + rng.normal(0, 1, n)).clip(2, 12)
    age = rng.integers(0, 80, n)
    quality = rng.integers(3, 10, n)
    garage = rng.integers(0, 4, n)
    basement = (area * rng.uniform(0.3, 0.9, n)).round()
    frontage = rng.normal(70, 20, n).clip(20, 200)
    price = (
        20_000 + 95 * area + 7_000 * rooms - 700 * age + 14_000 * quality
        + 6_000 * garage + 18 * basement
        + np.array([locations[l] for l in loc])
        + rng.normal(0, 22_000, n)
    )
    df = pd.DataFrame({"price": price.round(-2), "area": area.round(), "location": loc,
                       "rooms": rooms, "age": age, "quality": quality,
                       "garage": garage, "basement_area": basement,
                       "lot_frontage": frontage.round()})
    # sprinkle in some missing values so the cleaning step has something to do
    for col, frac in [("lot_frontage", 0.17), ("age", 0.03), ("rooms", 0.02)]:
        df.loc[rng.random(n) < frac, col] = np.nan
    return df


if DATA_PATH.exists():
    df = load_ames(DATA_PATH)
    print(f"Loaded data from {DATA_PATH}")
elif Path("house_prices_synthetic.csv").exists():
    df = load_ames(Path("house_prices_synthetic.csv"))
    print("Loaded house_prices_synthetic.csv (practice data, not the real Ames set).")
else:
    df = make_synthetic()
    print("No CSV found, generated synthetic data. Swap in the Kaggle file for the real thing.")

df.head()

## 2. Exploratory data analysis

Before any modelling I just want to look at the data: how big is it, what's missing, what do the numbers look like, and what does the thing I'm predicting look like?

In [ ]:
print("Shape:", df.shape)
df.info()

### Null check

In [ ]:
nulls = pd.DataFrame({
    "missing": df.isnull().sum(),
    "percent": (df.isnull().mean() * 100).round(2),
})
nulls[nulls["missing"] > 0].sort_values("missing", ascending=False)

Only a few columns have gaps, and `lot_frontage` is the biggest culprit. Nothing here is bad enough to justify dropping a column, so I'll fill the gaps rather than throw data away (more on that in step 4).

### Descriptive statistics

In [ ]:
df.describe().T.round(1)

Two things I look for in a table like this: values that look impossible (negative ages, zero-sized houses) and columns whose max is wildly bigger than the 75th percentile, which usually means outliers or skew.

### What does the target (price) look like?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(df["price"], bins=40, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Sale price")
axes[0].set_xlabel("Price")

sns.histplot(np.log1p(df["price"]), bins=40, kde=True, ax=axes[1], color="darkorange")
axes[1].set_title("log(1 + price)")
axes[1].set_xlabel("log price")

plt.tight_layout()
plt.show()

print(f"Skewness of price:      {df['price'].skew():.2f}")
print(f"Skewness of log(price): {np.log1p(df['price']).skew():.2f}")

House prices are almost always **right-skewed**: lots of modest homes and a long tail of expensive ones (the synthetic fallback data is more symmetric, so don't be surprised if you see that instead). A log transform tames that and often helps linear models. For this project I'm going to keep the target in plain dollars, because it makes the coefficients and errors easy to read ("this feature adds roughly $X"). If I wanted the best possible accuracy, modelling `log(price)` would be the first thing I'd try next.

## 3. Which features are likely predictors? (my reasoning)

Here's my thinking before I let any numbers talk:

- **`area` (living area):** probably the single strongest driver. More space costs more, and it's the first thing any buyer looks at.
- **`quality`:** two houses of the same size can differ massively in finish and materials. I expect a big effect.
- **`location` (neighbourhood):** the old "location, location, location" cliché exists for a reason. It's categorical, so it needs encoding, and I expect some neighbourhoods to carry a clear premium or discount.
- **`age`:** newer homes usually sell for more, so I expect a *negative* relationship, though renovated old houses can muddy it.
- **`rooms`:** likely positive, but it's tightly tied to `area` (bigger houses have more rooms). That overlap (multicollinearity) means its coefficient may look smaller than you'd guess, or even odd.
- **`basement_area` and `garage`:** extra usable space, so I expect modest positive effects.
- **`lot_frontage`:** the one I'm least sure about. I'll let the data decide.

I'll check all of these guesses against the correlation heatmap and the final coefficients. If the data disagrees with me, that's more interesting than if it agrees.

## 4. Missing values and categorical encoding

Two jobs here:

1. **Missing numbers:** I'll fill them with the **median** of the column. The median is a safer choice than the mean for skewed columns like frontage.
2. **Categorical text (`location`):** a linear model can't read neighbourhood names, so I'll **one-hot encode** them: one 0/1 column per neighbourhood. I drop one category as the baseline to avoid the "dummy variable trap" (perfectly redundant columns).

One thing I want to be careful about: **data leakage**. If I compute medians using the *whole* dataset, the test set quietly influences the training data. So for the actual model I'll bundle the imputing and encoding into a scikit-learn `Pipeline` that learns its medians from the training set only. For the heatmap below, which is just exploration, a quick fill on the full dataset is fine.

In [ ]:
numeric_features = ["area", "rooms", "age", "quality", "garage", "basement_area", "lot_frontage"]
categorical_features = ["location"]

# Quick-and-dirty cleaned copy, ONLY for exploration (the model uses a proper pipeline later)
explore = df.copy()
explore[numeric_features] = explore[numeric_features].fillna(explore[numeric_features].median())
explore = pd.get_dummies(explore, columns=categorical_features, drop_first=True, dtype=int)

print("Missing values left:", int(explore.isnull().sum().sum()))
explore.shape

## 5. Correlation heatmap

In [ ]:
corr = explore.corr()

# Full heatmap would be huge once every neighbourhood becomes a column, so I focus on the numeric features
num_corr = explore[["price"] + numeric_features].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(num_corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True, linewidths=0.5)
plt.title("Correlation between numeric features and price")
plt.show()

print("Correlation with price (all columns, strongest first):")
corr["price"].drop("price").sort_values(key=abs, ascending=False).head(10).round(3)

**What I take from this**

- The features most correlated with price are the ones I'd guessed: `quality` and `area` sit at the top, with `garage` and `basement_area` behind them. `age` goes the other way (older = cheaper), exactly as expected.
- Look at `area` vs. `rooms`: they're strongly correlated with each other. This is the multicollinearity I flagged earlier, and it's the reason I'll be cautious about reading the `rooms` coefficient in isolation.
- Correlation only captures *straight-line* relationships between two columns at a time. A feature can look weak here and still help once others are accounted for.

## 6. Train / test split (80/20)

I'll hold back 20% of the houses that the model never sees during training. That 20% is my honest check on whether the model works on houses it hasn't memorised. A fixed `random_state` keeps the split reproducible.

In [ ]:
X = df.drop(columns="price")
y = df["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

print(f"Train: {X_train.shape[0]} houses   |   Test: {X_test.shape[0]} houses")

## 7. Training the Linear Regression

The preprocessing pipeline does three things:

- numeric columns → fill gaps with the median, then **standardise** (mean 0, std 1)
- categorical column → one-hot encode (dropping the first category as the baseline)

Standardising doesn't change how accurate an ordinary linear regression is, but it makes the coefficients **comparable**. Without it, "per square foot" and "per quality point" live on totally different scales and you can't tell which feature matters more by looking at the numbers.

In [ ]:
def build_preprocessor():
    numeric_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])
    categorical_pipe = Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore", sparse_output=False)),
    ])
    return ColumnTransformer([
        ("num", numeric_pipe, numeric_features),
        ("cat", categorical_pipe, categorical_features),
    ])


lin_reg = Pipeline([
    ("prep", build_preprocessor()),
    ("model", LinearRegression()),
])

lin_reg.fit(X_train, y_train)
print("Model trained ✅")

## 8. Evaluation: MSE, RMSE and R²

- **MSE** averages the squared errors. It punishes big misses hard, but its units are "dollars squared", which is hard to picture.
- **RMSE** is the square root of MSE, so it's back in dollars: roughly "how far off is a typical prediction?"
- **R²** is the share of the variation in price that the model explains. 1.0 is perfect, 0 means "no better than always guessing the average price".

I'll report train *and* test. If train is much better than test, that's a sign of overfitting.

In [ ]:
def evaluate(model, name="Model"):
    rows = {}
    for label, X_, y_ in [("Train", X_train, y_train), ("Test", X_test, y_test)]:
        pred = model.predict(X_)
        mse = mean_squared_error(y_, pred)
        rows[label] = {"MSE": mse, "RMSE": np.sqrt(mse), "R²": r2_score(y_, pred)}
    out = pd.DataFrame(rows).T
    out.index = [f"{name} – {i}" for i in out.index]
    return out


results = evaluate(lin_reg, "Linear Regression")
results.style.format({"MSE": "{:,.0f}", "RMSE": "${:,.0f}", "R²": "{:.3f}"})

In [ ]:
test_rmse = results.loc["Linear Regression – Test", "RMSE"]
test_r2 = results.loc["Linear Regression – Test", "R²"]
print(f"On houses it has never seen, the model explains about {test_r2:.0%} of the variation in price,")
print(f"and a typical prediction misses by roughly ${test_rmse:,.0f}.")
print(f"For context, the average house price is ${y.mean():,.0f}, so that's about {test_rmse / y.mean():.0%} of an average price.")

**How I read this:** I look at the gap between train and test first. If they're close, the model generalises. A large gap would mean it's memorising. Then I look at RMSE relative to the average price, because a "$30k error" is a disaster on a $90k house but fine on a $600k one.

## 9. Actual vs. predicted

If the model were perfect, every dot would sit on the diagonal line. The tighter the cloud hugs the line, the better.

In [ ]:
y_pred = lin_reg.predict(X_test)

plt.figure(figsize=(6.5, 6))
plt.scatter(y_test, y_pred, alpha=0.5, edgecolor="none", color="steelblue")
lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
plt.plot(lims, lims, "r--", lw=2, label="Perfect prediction")
plt.xlabel("Actual price")
plt.ylabel("Predicted price")
plt.title("Actual vs. predicted (test set)")
plt.legend()
plt.tight_layout()
plt.show()

Things worth noticing: does the cloud get *wider* for expensive houses? Does the model consistently over-predict cheap homes or under-predict pricey ones? Any predicted prices that are negative or absurdly low would be a classic linear-model giveaway.

## 10. Residual analysis

A residual is `actual − predicted`. If a linear model is a good fit, the residuals should look like **random noise**: scattered evenly around zero with no pattern. A curve, a funnel shape or a clump means the model is missing something systematic.

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_pred, residuals, alpha=0.5, edgecolor="none", color="seagreen")
axes[0].axhline(0, color="red", ls="--", lw=2)
axes[0].set_xlabel("Predicted price")
axes[0].set_ylabel("Residual (actual − predicted)")
axes[0].set_title("Residuals vs. predicted")

sns.histplot(residuals, bins=40, kde=True, ax=axes[1], color="seagreen")
axes[1].axvline(0, color="red", ls="--", lw=2)
axes[1].set_xlabel("Residual")
axes[1].set_title("Distribution of residuals")

plt.tight_layout()
plt.show()

print(f"Mean residual: ${residuals.mean():,.0f}  (should be close to 0)")

**My honest read:** the histogram should be roughly bell-shaped and centred on zero. In the left plot, I'm hunting for a **funnel** (errors growing with price, called *heteroscedasticity*) or a **curve** (a non-linear relationship the straight-line model can't capture). On real housing data it's very common to see some funnel shape, because a 10% error on an expensive house is a bigger dollar error than on a cheap one. That's the textbook argument for modelling `log(price)`, and it's my first idea for improvement.

## 11. Coefficient analysis: what actually moves the price?

Because the numeric features were standardised, each coefficient reads as: *"the change in price (in dollars) for a one-standard-deviation increase in this feature, holding everything else fixed."* That makes them directly comparable.

For the `location_...` columns, the coefficient is the price difference **relative to the baseline neighbourhood** (the one that was dropped during encoding).

In [ ]:
feature_names = lin_reg.named_steps["prep"].get_feature_names_out()
feature_names = [n.split("__", 1)[1] for n in feature_names]

coefs = pd.Series(lin_reg.named_steps["model"].coef_, index=feature_names).sort_values()

fig, ax = plt.subplots(figsize=(9, max(5, 0.3 * len(coefs))))
colors = ["firebrick" if c < 0 else "seagreen" for c in coefs]
coefs.plot.barh(ax=ax, color=colors)
ax.axvline(0, color="black", lw=1)
ax.set_title("Linear Regression coefficients (effect on price, $)")
ax.set_xlabel("Change in price")
plt.tight_layout()
plt.show()

print("Biggest positive effects:")
print(coefs.sort_values(ascending=False).head(5).round(0).to_string())
print("\nBiggest negative effects:")
print(coefs.sort_values().head(5).round(0).to_string())

**Reading it like a human**

- The tall green bars are the features that push price **up**; the red bars pull it **down**. Compare them against my predictions in step 3. Where I was right, great; where I wasn't, it's worth asking why.
- Remember these are *associations in this dataset*, not proof of cause. Adding a garage won't automatically add the exact coefficient to your sale price.
- If `rooms` looks small or even negative, don't panic: it overlaps heavily with `area`, so once area is accounted for, extra rooms add little *on top*. That's multicollinearity in action.
- Location effects depend on which neighbourhood was chosen as the baseline. The *differences between* neighbourhoods are the meaningful part.

## 12. Bonus: Ridge and Lasso

Plain linear regression can get jumpy when features overlap (like `area` and `rooms`) or when there are lots of one-hot columns. **Regularisation** adds a penalty on large coefficients:

- **Ridge (L2)** shrinks all coefficients toward zero but rarely makes them exactly zero.
- **Lasso (L1)** can push weak coefficients all the way to zero, so it doubles as automatic feature selection.

I'm using the `CV` versions, which pick the penalty strength themselves via cross-validation on the training set, so I'm not tuning on the test set.

In [ ]:
ridge = Pipeline([
    ("prep", build_preprocessor()),
    ("model", RidgeCV(alphas=np.logspace(-3, 3, 50))),
]).fit(X_train, y_train)

lasso = Pipeline([
    ("prep", build_preprocessor()),
    ("model", LassoCV(cv=5, random_state=RANDOM_STATE, max_iter=20_000)),
]).fit(X_train, y_train)

comparison = pd.concat([
    evaluate(lin_reg, "Linear"),
    evaluate(ridge, "Ridge"),
    evaluate(lasso, "Lasso"),
])

print(f"Ridge picked alpha = {ridge.named_steps['model'].alpha_:.3g}")
print(f"Lasso picked alpha = {lasso.named_steps['model'].alpha_:.3g}")

comparison.style.format({"MSE": "{:,.0f}", "RMSE": "${:,.0f}", "R²": "{:.3f}"})

In [ ]:
coef_table = pd.DataFrame({
    "Linear": lin_reg.named_steps["model"].coef_,
    "Ridge": ridge.named_steps["model"].coef_,
    "Lasso": lasso.named_steps["model"].coef_,
}, index=feature_names)

n_zero = (coef_table["Lasso"].abs() < 1e-6).sum()
print(f"Lasso set {n_zero} of {len(coef_table)} coefficients to exactly zero.\n")

coef_table.round(0).sort_values("Linear", key=abs, ascending=False).head(12)

**Takeaway:** on a clean, well-behaved dataset like this, regularisation usually gives only a *small* improvement, and sometimes none, because with ~1,000+ rows and a handful of features, ordinary least squares isn't struggling much. Its real value shows up when you have many more features, or when features are heavily correlated. If Lasso zeroes out a few coefficients, that's a hint about which features the model considers redundant.

## 13. Wrap-up

**What I did:** loaded and explored the data, reasoned about features before modelling, cleaned and encoded inside a leak-free pipeline, split 80/20, trained a Linear Regression, and checked it with MSE / RMSE / R², an actual-vs-predicted plot, a residual plot and coefficient analysis, then compared it with Ridge and Lasso.

**What I'd do next**

1. Model `log(price)` to handle the skew and the funnel-shaped residuals, then convert predictions back to dollars.
2. Add more of the 80 Ames columns (kitchen quality, lot size, year remodelled) and let Lasso sort out which ones matter.
3. Look at the biggest individual misses. They usually reveal outliers or data-entry quirks worth investigating.
4. Try interaction terms (e.g. `area × quality`) since a big house of high quality is probably worth more than the two effects added up.
5. Move from a single split to k-fold cross-validation for a more trustworthy performance estimate.

**Honest limitations:** linear regression assumes straight-line relationships, roughly constant error spread and not-too-correlated features. Real housing data bends all three rules a bit, which is exactly why the diagnostics in steps 9–11 matter more than the headline R² number.

---
*References: scikit-learn docs for `LinearRegression`, `mean_squared_error`, `r2_score`, `Ridge`, `Lasso`; the Kaggle "House Prices: Advanced Regression Techniques" competition page.*